# NB02 — Surface Extraction & Pivot to Wide Format

**Objective**:
1. Read all valid `output_data_points.csv` from ZIP archives (long format, one row = one measurement)
2. Extract the surface soil layer (`upper_depth_cm <= 10`)
3. Where multiple depth intervals exist for the same station × property, apply depth-weighted mean
4. Pivot from long format to wide format (one row = one station, each property = one column)
5. Deduplicate stations that appear in multiple ZIPs for the same country

**Inputs**:
- `data/country/*` (ZIP archives)
- `data/intermediate/audit/zip_inventory.csv` (from NB01 — only valid ZIPs used)

**Outputs**:
- `data/intermediate/wosis_surface_long.parquet`  — surface-filtered long format
- `data/intermediate/wosis_surface_wide.parquet`  — one row per unique station
- `data/intermediate/wosis_surface_flags.parquet` — per station × property: depth_method, n_depth_intervals
- `logs/nb02_surface_extraction.log`

**STOP conditions**:
- Output station count < 80% of NB01 expected total → STOP
- Pivot produces unexpected columns → STOP
- Any ZIP listed as valid in NB01 fails to load → ERROR logged, counted

In [20]:
import zipfile
import io
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# ── Paths ──────────────────────────────────────────────────────────────────
BASE        = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
COUNTRY_DIR = BASE / 'data' / 'country'
INTER_DIR   = BASE / 'data' / 'intermediate'
AUDIT_DIR   = INTER_DIR / 'audit'
LOG_DIR     = BASE / 'logs'

# ── Logging ────────────────────────────────────────────────────────────────
log_path = LOG_DIR / 'nb02_surface_extraction.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_path, mode='w'),
        logging.StreamHandler()
    ]
)
log = logging.getLogger('nb02')
log.info(f'NB02 started at {datetime.now().isoformat()}')

# ── Constants ──────────────────────────────────────────────────────────────
SURFACE_UPPER_DEPTH_MAX = 10   # cm — rows with upper_depth_cm > this are discarded

# Canonical property list — 20 WoSIS standard + 4 discovered in NB01 (Australia ZIP)
PROPERTIES = [
    'Al', 'BD', 'Ca', 'CaCO3', 'CEC', 'CF', 'clay', 'Cu', 'EC', 'Fe',
    'K', 'Mg', 'Mn', 'N', 'Na', 'nematode', 'occ', 'P', 'pH', 'sand',
    'silt', 'TC', 'WR_gravimetric', 'WR_volumetric'
]

CSV_COLS = [
    'lat', 'lon', 'property', 'original_name',
    'upper_depth_cm', 'lower_depth_cm', 'value',
    'unit', 'sampling_date', 'data_source'
]

2026-03-19 18:24:17,202 [INFO] NB02 started at 2026-03-19T18:24:17.202864


In [21]:
# ── Load NB01 inventory — use only ZIPs with no error and has_csv=True ─────
inventory = pd.read_csv(AUDIT_DIR / 'zip_inventory.csv')

valid_csv_zips = inventory[
    (inventory['has_csv'] == True) &
    (inventory['error'].isna())
].copy()

log.info(f'NB01 inventory loaded: {len(inventory)} total ZIPs')
log.info(f'  Valid ZIPs with CSV data : {len(valid_csv_zips)}')
log.info(f'  Skipped (no CSV)         : {(inventory["has_csv"]==False).sum()}')
log.info(f'  Skipped (BadZipFile)     : {inventory["error"].notna().sum()}')

expected_total_stations = valid_csv_zips['n_stations'].sum()
log.info(f'Expected stations from NB01 (with duplicates): {expected_total_stations:,}')

print(f'Valid ZIPs to process : {len(valid_csv_zips)}')
print(f'Expected stations     : {expected_total_stations:,} (before deduplication)')

2026-03-19 18:24:18,580 [INFO] NB01 inventory loaded: 118 total ZIPs
2026-03-19 18:24:18,580 [INFO]   Valid ZIPs with CSV data : 77
2026-03-19 18:24:18,581 [INFO]   Skipped (no CSV)         : 41
2026-03-19 18:24:18,582 [INFO]   Skipped (BadZipFile)     : 12
2026-03-19 18:24:18,582 [INFO] Expected stations from NB01 (with duplicates): 106,548


Valid ZIPs to process : 77
Expected stations     : 106,548 (before deduplication)


In [22]:
# ── Load all CSVs, filter surface layer, attach country metadata ───────────
surface_chunks = []
load_errors    = []

for _, inv_row in valid_csv_zips.iterrows():
    zip_hash = inv_row['zip_hash']
    zip_path = COUNTRY_DIR / str(zip_hash)
    country  = inv_row['country']
    iso3     = inv_row['iso3']

    try:
        with zipfile.ZipFile(zip_path, 'r') as zf:
            with zf.open('output_data_points.csv') as f:
                df = pd.read_csv(
                    io.TextIOWrapper(f, encoding='utf-8'),
                    low_memory=False,
                    usecols=CSV_COLS
                )

        # Cast numeric columns
        df['upper_depth_cm'] = pd.to_numeric(df['upper_depth_cm'], errors='coerce')
        df['lower_depth_cm'] = pd.to_numeric(df['lower_depth_cm'], errors='coerce')
        df['value']          = pd.to_numeric(df['value'],          errors='coerce')
        df['lat']            = pd.to_numeric(df['lat'],            errors='coerce')
        df['lon']            = pd.to_numeric(df['lon'],            errors='coerce')

        # Attach origin metadata
        df['zip_hash'] = zip_hash
        df['country']  = country
        df['iso3']     = iso3

        # Surface filter: keep only records where upper_depth <= threshold
        surface = df[df['upper_depth_cm'] <= SURFACE_UPPER_DEPTH_MAX].copy()

        n_all     = df[['lat', 'lon']].drop_duplicates().shape[0]
        n_surface = surface[['lat', 'lon']].drop_duplicates().shape[0]
        pct_surf  = 100 * n_surface / n_all if n_all > 0 else 0
        log.info(
            f'  {zip_hash} ({country}): {n_all} stations, '
            f'{n_surface} with surface records ({pct_surf:.0f}%)'
        )

        surface_chunks.append(surface)

    except Exception as e:
        msg = f'ERROR loading {zip_hash} ({country}): {e}'
        log.error(msg)
        load_errors.append(msg)

if load_errors:
    print(f'\n{len(load_errors)} load error(s):')
    for e in load_errors:
        print(f'  {e}')

log.info('Concatenating surface chunks...')
df_long = pd.concat(surface_chunks, ignore_index=True)
log.info(f'Combined surface table: {len(df_long):,} raw rows from {len(surface_chunks)} ZIPs')
print(f'Combined surface table: {len(df_long):,} raw rows')

2026-03-19 18:24:29,895 [INFO]   223krk83 (Albania): 83 stations, 83 with surface records (100%)
2026-03-19 18:24:30,105 [INFO]   224lf9p7 (Greece): 338 stations, 330 with surface records (98%)
2026-03-19 18:24:30,149 [INFO]   224nbt59 (Greece): 338 stations, 330 with surface records (98%)
2026-03-19 18:24:30,262 [INFO]   227yj7zt (France): 3127 stations, 3122 with surface records (100%)
2026-03-19 18:24:30,302 [INFO]   229at92m (Georgia): 18 stations, 18 with surface records (100%)
2026-03-19 18:24:30,332 [INFO]   22h6n8tu (Papua New Guinea): 16 stations, 16 with surface records (100%)
2026-03-19 18:24:30,402 [INFO]   22t5yo8e (Burkina Faso): 2116 stations, 2109 with surface records (100%)
2026-03-19 18:24:30,471 [INFO]   23jo4hd3 (Burundi): 29 stations, 28 with surface records (97%)
2026-03-19 18:24:30,572 [INFO]   242jncwn (Cameroon): 1394 stations, 1387 with surface records (99%)
2026-03-19 18:24:30,651 [INFO]   2456acex (Netherlands): 1354 stations, 1300 with surface records (96%)

Combined surface table: 806,504 raw rows


In [23]:
# ── Remove exact duplicate rows ────────────────────────────────────────────
# Source: internal duplicates within the same data_source (e.g. WoSIS).
# No cross-source duplication has been observed in the data.
#
# RULE: Remove rows that are fully identical on all measurement dimensions:
#   same station (lat/lon) + same property + same depth interval
#   + same sampling date + same value
#
# Rows with the same location and property but different years or different
# values are NOT removed — they represent valid temporal observations.

dedup_keys = [
    'lat', 'lon', 'iso3', 'property',
    'upper_depth_cm', 'lower_depth_cm',
    'sampling_date',
    'value'
]

n_before = len(df_long)
df_long  = df_long.drop_duplicates(subset=dedup_keys, keep='first').reset_index(drop=True)
n_after  = len(df_long)
n_removed = n_before - n_after
pct_removed = 100 * n_removed / n_before if n_before > 0 else 0

log.info(
    f'Exact-duplicate removal: {n_before:,} → {n_after:,} rows '
    f'(removed {n_removed:,} rows, {pct_removed:.1f}%)'
)
print(f'Exact-duplicate removal: {n_before:,} → {n_after:,} rows')
print(f'  Removed : {n_removed:,} ({pct_removed:.1f}%) — internal source duplicates')
print(f'  Retained: {n_after:,} ({100 - pct_removed:.1f}%)')

2026-03-19 18:24:37,696 [INFO] Exact-duplicate removal: 806,504 → 367,243 rows (removed 439,261 rows, 54.5%)


Exact-duplicate removal: 806,504 → 367,243 rows
  Removed : 439,261 (54.5%) — internal source duplicates
  Retained: 367,243 (45.5%)


In [24]:
# ── Temporal strategy then depth-weighted mean ────────────────────────────
#
# After exact-dedup, a station × property × depth can still have multiple rows
# from different years (temporal series). We need one value per station × property
# for the wide table.
#
# TEMPORAL RULE:
#   Use the MOST RECENT measurement per (lat, lon, iso3, property, depth interval).
#   Rationale: soil properties evolve over time; most recent is most representative.
#   All years are preserved in wosis_surface_long.parquet for any temporal analysis.
#
# DEPTH RULE:
#   After selecting the most recent temporal value, apply depth-weighted mean
#   across multiple depth intervals (e.g. 0-5 + 5-10 cm → 0-10 cm representative).

# Step 1 — select most recent observation per (station, property, depth interval)
df_long['sampling_date'] = pd.to_datetime(df_long['sampling_date'], errors='coerce')

TEMPORAL_KEYS = ['lat', 'lon', 'iso3', 'property', 'upper_depth_cm', 'lower_depth_cm']

df_most_recent = (
    df_long
    .sort_values('sampling_date', ascending=True, na_position='first')
    .drop_duplicates(subset=TEMPORAL_KEYS, keep='last')   # keep latest date
    .reset_index(drop=True)
)

n_temporal_removed = len(df_long) - len(df_most_recent)
log.info(f'Temporal selection (most recent per depth interval): '
         f'{len(df_long):,} → {len(df_most_recent):,} rows '
         f'(removed {n_temporal_removed:,} older-year duplicates)')
print(f'Temporal selection: kept most recent per station×property×depth')
print(f'  {len(df_long):,} → {len(df_most_recent):,} rows ({n_temporal_removed:,} older records set aside)')
print(f'  All years preserved in wosis_surface_long.parquet')

# Step 2 — depth-weighted mean across depth intervals
def depth_weighted_mean(group: pd.DataFrame) -> pd.Series:
    g = group.dropna(subset=['value', 'upper_depth_cm', 'lower_depth_cm']).copy()
    if len(g) == 0:
        return pd.Series({
            'value': np.nan,
            'depth_method': 'no_valid_obs',
            'n_depth_intervals': 0,
            'eff_upper_cm': np.nan,
            'eff_lower_cm': np.nan,
            'data_source': np.nan,
            'sampling_date': pd.NaT,
        })

    g['thickness'] = g['lower_depth_cm'] - g['upper_depth_cm']
    g = g[g['thickness'] > 0]

    if len(g) == 0:
        return pd.Series({
            'value': np.nan,
            'depth_method': 'zero_thickness',
            'n_depth_intervals': 0,
            'eff_upper_cm': np.nan,
            'eff_lower_cm': np.nan,
            'data_source': np.nan,
            'sampling_date': pd.NaT,
        })

    total_thickness = g['thickness'].sum()
    weighted_value  = (g['value'] * g['thickness']).sum() / total_thickness
    method          = 'single' if len(g) == 1 else 'depth_weighted_mean'

    return pd.Series({
        'value': weighted_value,
        'depth_method': method,
        'n_depth_intervals': len(g),
        'eff_upper_cm': g['upper_depth_cm'].min(),
        'eff_lower_cm': g['lower_depth_cm'].max(),
        'data_source': '|'.join(sorted(g['data_source'].dropna().unique())),
        'sampling_date': g['sampling_date'].max(),   # most recent date kept
    })

GROUP_KEYS = ['lat', 'lon', 'iso3', 'country', 'property']

log.info('Computing depth-weighted means per station × property...')
agg = (
    df_most_recent
    .groupby(GROUP_KEYS, sort=False)
    .apply(depth_weighted_mean, include_groups=False)
    .reset_index()
)
log.info(f'Aggregated: {len(agg):,} station × property rows')
print(f'Aggregated: {len(agg):,} station × property rows')

2026-03-19 18:24:50,782 [INFO] Temporal selection (most recent per depth interval): 367,243 → 320,858 rows (removed 46,385 older-year duplicates)
2026-03-19 18:24:50,783 [INFO] Computing depth-weighted means per station × property...


Temporal selection: kept most recent per station×property×depth
  367,243 → 320,858 rows (46,385 older records set aside)
  All years preserved in wosis_surface_long.parquet


2026-03-19 18:32:13,031 [INFO] Aggregated: 239,336 station × property rows


Aggregated: 239,336 station × property rows


In [27]:
# ── Separate flags from values before pivoting ─────────────────────────────
df_flags = agg[[
    'lat', 'lon', 'iso3', 'country', 'property',
    'depth_method', 'n_depth_intervals', 'eff_upper_cm', 'eff_lower_cm',
    'data_source', 'sampling_date'
]].copy()

# ── Pivot values to wide format ───────────────────────────────────────────
log.info('Pivoting to wide format...')

df_wide = agg.pivot_table(
    index=['lat', 'lon', 'iso3', 'country'],
    columns='property',
    values='value',
    aggfunc='first'
).reset_index()

df_wide.columns.name = None

# Ensure all 20 expected property columns are present
for prop in PROPERTIES:
    if prop not in df_wide.columns:
        df_wide[prop] = np.nan
        log.warning(f'Property "{prop}" absent from all ZIPs — column added as NaN')

# Reorder: identity cols first, then properties alphabetically
id_cols   = ['lat', 'lon', 'iso3', 'country']
prop_cols = sorted([c for c in df_wide.columns if c in PROPERTIES])
extra     = [c for c in df_wide.columns if c not in id_cols + prop_cols]
df_wide   = df_wide[id_cols + prop_cols + extra]

# Stable station_id
df_wide = df_wide.reset_index(drop=True)
df_wide.insert(0, 'station_id', [
    f"{row.iso3}_{i:06d}" for i, row in enumerate(df_wide.itertuples())
])

log.info(f'Wide table: {len(df_wide):,} stations × {len(df_wide.columns)} columns')
print(f'Wide table: {len(df_wide):,} stations × {len(df_wide.columns)} columns')

2026-03-19 19:48:00,257 [INFO] Pivoting to wide format...


2026-03-19 19:48:00,527 [INFO] Wide table: 53,708 stations × 29 columns


Wide table: 53,708 stations × 29 columns


In [28]:
n_output = len(df_wide)
# expected_total_stations counts duplicates; unique is what we compare against
unique_expected = valid_csv_zips.drop_duplicates(subset='iso3')['n_stations'].sum()
pct = 100 * n_output / unique_expected if unique_expected > 0 else 0

print(f'Unique-country expected stations : {unique_expected:,}')
print(f'Output stations (wide)           : {n_output:,}')
print(f'Recovery rate                    : {pct:.1f}%')

# Note: recovery can exceed 100% when multi-region ZIPs cover non-overlapping areas
# of the same country — this is expected and OK.
if pct < 80:
    raise AssertionError(
        f'STOP: Only {pct:.1f}% of expected stations recovered '
        f'(threshold=80%). Check surface filter and CSV loading.'
    )

# 2. Unexpected columns
extra_cols = [c for c in df_wide.columns if c not in PROPERTIES + id_cols + ['station_id']]
if extra_cols:
    log.warning(f'Unexpected columns in wide table: {extra_cols}')
    print(f'WARNING: Unexpected columns: {extra_cols}')

Unique-country expected stations : 56,991
Output stations (wide)           : 53,708
Recovery rate                    : 94.2%


In [29]:
# ── Summary statistics ─────────────────────────────────────────────────────
print('=== NB02 SUMMARY ===')
print(f'Raw surface rows loaded             : {n_before:,}')
print(f'After exact-duplicate removal       : {n_after:,}  (−{n_removed:,}, {pct_removed:.1f}%)')
print(f'After temporal selection            : {len(df_most_recent):,}  (−{n_temporal_removed:,} older-year records)')
print(f'Unique stations (wide table)        : {len(df_wide):,}')
print(f'Countries                           : {df_wide["iso3"].nunique()}')
print()
print('Property completeness (% stations with at least one surface observation):')
for prop in sorted(PROPERTIES):
    if prop in df_wide.columns:
        pct_prop = 100 * df_wide[prop].notna().mean()
        bar = '#' * int(pct_prop / 5)
        print(f'  {prop:20s} {pct_prop:5.1f}%  {bar}')

print()
print('Stations per country:')
print(
    df_wide.groupby(['iso3', 'country'])
    .size()
    .reset_index(name='n_stations')
    .sort_values('n_stations', ascending=False)
    .to_string(index=False)
)

=== NB02 SUMMARY ===
Raw surface rows loaded             : 806,504
After exact-duplicate removal       : 367,243  (−439,261, 54.5%)
After temporal selection            : 320,858  (−46,385 older-year records)
Unique stations (wide table)        : 53,708
Countries                           : 23

Property completeness (% stations with at least one surface observation):
  Al                     2.7%  
  BD                     1.1%  
  CEC                   22.9%  ####
  CF                    17.4%  ###
  Ca                     2.8%  
  CaCO3                 11.9%  ##
  Cu                     2.7%  
  EC                    40.3%  ########
  Fe                     2.7%  
  K                      2.8%  
  Mg                     2.8%  
  Mn                     2.7%  
  N                     31.7%  ######
  Na                     0.1%  
  P                     18.4%  ###
  TC                     6.5%  #
  WR_gravimetric         9.0%  #
  WR_volumetric          0.8%  
  clay                  54.

In [30]:
# ── Save outputs ───────────────────────────────────────────────────────────
df_long.to_parquet(INTER_DIR / 'wosis_surface_long.parquet',  index=False)
df_wide.to_parquet(INTER_DIR / 'wosis_surface_wide.parquet',  index=False)
df_flags.to_parquet(INTER_DIR / 'wosis_surface_flags.parquet', index=False)

log.info(f'Saved wosis_surface_long.parquet  — {len(df_long):,} rows')
log.info(f'Saved wosis_surface_wide.parquet  — {len(df_wide):,} rows')
log.info(f'Saved wosis_surface_flags.parquet — {len(df_flags):,} rows')
log.info('NB02 completed successfully — proceed to NB03')

print('\nOutputs saved:')
for fname in ['wosis_surface_long.parquet', 'wosis_surface_wide.parquet', 'wosis_surface_flags.parquet']:
    p = INTER_DIR / fname
    print(f'  {fname}: {p.stat().st_size / 1e6:.2f} MB')

2026-03-19 19:53:20,454 [INFO] Saved wosis_surface_long.parquet  — 367,243 rows
2026-03-19 19:53:20,455 [INFO] Saved wosis_surface_wide.parquet  — 53,708 rows
2026-03-19 19:53:20,455 [INFO] Saved wosis_surface_flags.parquet — 239,336 rows
2026-03-19 19:53:20,456 [INFO] NB02 completed successfully — proceed to NB03



Outputs saved:
  wosis_surface_long.parquet: 2.86 MB
  wosis_surface_wide.parquet: 2.16 MB
  wosis_surface_flags.parquet: 2.00 MB
